# TD01 — Perceptron

## Objectif

Le perceptron (Rosenblatt, 1957) est le plus simple des réseaux de neurones : **un seul neurone** qui sépare l'espace en deux. En dimension 2, il trace une **droite** qui partage le plan en deux demi-plans. On l'implémente, on le fait **apprendre** avec la règle de Rosenblatt, puis on **généralise** à une dimension quelconque (cours « 2 - Perceptron »).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from types import SimpleNamespace

## 1. Le modèle de neurone

Un perceptron en dimension 2 possède :
- deux entrées $x_1, x_2$ (un point du plan) ;
- trois poids $w_0$ (le **biais**), $w_1$, $w_2$ ;
- une **fonction d'activation** : la fonction de Heaviside

$$H(s)=\begin{cases}0 & \text{si } s < 0 \\ 1 & \text{si } s \ge 0\end{cases}$$

Le neurone calcule $s = x_1 w_1 + x_2 w_2 - w_0$, puis renvoie $H(s)$ : il sépare le plan en deux demi-plans, séparés par la **droite** d'équation $x_1 w_1 + x_2 w_2 - w_0 = 0$.

> **Exercice 1** — Pour chacun des jeux de poids suivants, déterminez (à la main) sur quelle partie du plan le neurone est **activé** (sortie 1) :
>
> 1. $w_0 = 0,\ w_1 = 1,\ w_2 = 1$
> 2. $w_0 = 0,\ w_1 = -1,\ w_2 = 2$
> 3. $w_0 = 0,\ w_1 = 0,\ w_2 = 1$
> 4. $w_0 = 1,\ w_1 = 1,\ w_2 = 1$
> 5. $w_0 = -1,\ w_1 = 1,\ w_2 = 1$
>
> **Indice** : intéressez-vous à la droite frontière d'équation $x_1 w_1 + x_2 w_2 - w_0 = 0$.

> **Exercice 2** — Quel serait le problème si l'on n'avait pas introduit le **biais** (le poids $w_0$) ?

> [!note] Indice
> Sans $w_0$, la droite frontière passe toujours par l'origine $(0, 0)$.

## 2. Utilitaires d'affichage

Ces fonctions visualisent un perceptron (son demi-plan activé et sa droite frontière) ainsi qu'un jeu de données d'apprentissage. Elles servent aussi à **vérifier** l'exercice 1.

In [ ]:
def make_axis():
    fig, ax = plt.subplots()
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal', adjustable='box')
    ax.axhline(0, color='black', linewidth=1)
    ax.axvline(0, color='black', linewidth=1)
    ax.grid()
    return ax


def display_plane(p, ax=None):
    """Affiche la droite de séparation et colorie les demi-plans.

    Bleu : demi-plan où le perceptron sort 1.
    Orange : demi-plan où le perceptron sort 0.
    """
    if ax is None:
        ax = make_axis()
    xx, yy = np.meshgrid(np.linspace(-2, 2, 400), np.linspace(-2, 2, 400))
    zz = (xx * p.w1 + yy * p.w2 - p.w0 >= 0).astype(int)
    ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=['tab:orange', 'tab:blue'], alpha=0.1)
    if p.w2 != 0:
        ax.axline((0, p.w0 / p.w2), (1, (p.w0 - p.w1) / p.w2), color='black', linewidth=2)
    elif p.w1 != 0:
        ax.axvline(p.w0 / p.w1, color='black', linewidth=2)


@dataclass
class TrainPoint:
    x1: float
    x2: float
    t: int


def display_train(points, ax=None):
    if ax is None:
        ax = make_axis()
    blue = [(p.x1, p.x2) for p in points if p.t == 1]
    orange = [(p.x1, p.x2) for p in points if p.t == 0]
    if blue:
        ax.plot(*zip(*blue), 'o', color='blue')
    if orange:
        ax.plot(*zip(*orange), 'o', color='orange')

### Vérification de l'exercice 1

Affichez les 5 perceptrons de l'exercice 1 et vérifiez vos réponses.

In [ ]:
configs = [
    ('w0=0, w1=1, w2=1',   0,  1, 1),
    ('w0=0, w1=-1, w2=2',  0, -1, 2),
    ('w0=0, w1=0, w2=1',   0,  0, 1),
    ('w0=1, w1=1, w2=1',   1,  1, 1),
    ('w0=-1, w1=1, w2=1', -1,  1, 1),
]
for label, w0, w1, w2 in configs:
    ax = make_axis()
    display_plane(SimpleNamespace(w0=w0, w1=w1, w2=w2), ax)
    ax.set_title(label)
    plt.show()

## 3. Implémentation du perceptron (2D)

> **Exercice 3** — Implémentez le neurone avec une classe `Perceptron` : la méthode `output(x1, x2)` renvoie `1` si $x_1 w_1 + x_2 w_2 - w_0 \ge 0$, sinon `0`.

> **Exercice 4** — Ajoutez la méthode `learn(self, x1, x2, t, r=0.1)` qui effectue **une** étape d'apprentissage. Si la sortie $y$ diffère de la cible $t$, on met à jour chaque poids : $w_i \leftarrow w_i - r\,(y - t)\,x_i$, avec $x_0 = -1$ pour le biais.

> **Exercice 5** — Ajoutez la méthode `train(self, points, r=0.1, max_epochs=100)` qui répète l'apprentissage sur tous les points **tant que les poids changent** (avec une borne de sécurité).

In [ ]:
@dataclass
class Perceptron:
    w0: float  # biais
    w1: float
    w2: float

    def __repr__(self):
        return f"Perceptron({self.w0}, {self.w1}, {self.w2})"

    def output(self, x1, x2):
        # BEGIN
        # raise NotImplementedError
        if x1 * self.w1 + x2 * self.w2 - self.w0 >= 0:
            return 1
        return 0
        # END

    def learn(self, x1, x2, t, r=0.1):
        # BEGIN
        # raise NotImplementedError
        ## Indice : y = self.output(x1, x2) ; si y != t, w_i <- w_i - r*(y - t)*x_i
        y = self.output(x1, x2)
        if y != t:
            self.w0 += r * (t - y) * (-1)
            self.w1 += r * (t - y) * x1
            self.w2 += r * (t - y) * x2
        # END

    def train(self, points, r=0.1, max_epochs=100):
        # BEGIN
        # raise NotImplementedError
        ## Indice : boucle sur tous les points, tant que les poids changent
        for _ in range(max_epochs):
            changed = False
            for pt in points:
                w_before = (self.w0, self.w1, self.w2)
                self.learn(pt.x1, pt.x2, pt.t, r)
                if (self.w0, self.w1, self.w2) != w_before:
                    changed = True
            if not changed:
                break
        # END

### Vérification de `output`

On vérifie la sortie du neurone sur quelques cas choisis (les 5 configurations de l'exercice 1).

In [ ]:
def check_output():
    cases = [
        (Perceptron(0, 1, 1),  (1, 1),  1),
        (Perceptron(0, 1, 1),  (-1, -1), 0),
        (Perceptron(0, -1, 2), (1, 0),  0),
        (Perceptron(0, -1, 2), (0, 1),  1),
        (Perceptron(0, 0, 1),  (0, -1), 0),
        (Perceptron(0, 0, 1),  (0, 1),  1),
        (Perceptron(1, 1, 1),  (0, 0),  0),
        (Perceptron(1, 1, 1),  (1, 1),  1),
        (Perceptron(-1, 1, 1), (0, 0),  1),
        (Perceptron(-1, 1, 1), (-2, -2), 0),
    ]
    for p, (x1, x2), expected in cases:
        assert p.output(x1, x2) == expected, f"{p} : output({x1}, {x2}) != {expected}"
    print("output check — OK")


check_output()

### Jeux de données d'entraînement

Les 4 jeux du cours (exercice 4) : le ET logique, une droite oblique, un jeu séparable, et le **XOR** (qui n'est pas linéairement séparable).

In [ ]:
def make_points(rows):
    return [TrainPoint(*r) for r in rows]


entree_1 = make_points([(1, 1, 1), (1, 0, 0), (0, 1, 0), (0, 0, 0)])  # ET logique (AND)
entree_2 = make_points([(-1, -1, 1), (1, 1, 0)])                      # droite oblique
entree_3 = make_points([
    (-1.5, 0.2, 1), (-1, 0.5, 1), (-0.5, 1, 1),
    (-1.5, -1, 0), (-1, -0.6, 0), (-0.5, 0, 0),
])
entree_4 = make_points([(1, 1, 0), (1, 0, 1), (0, 1, 1), (0, 0, 0)])  # XOR : non séparable

### Vérification de `learn`

Une étape d'apprentissage doit mettre à jour les poids exactement comme la règle de Rosenblatt.

In [ ]:
def check_learn():
    p = Perceptron(0, 0, 0)
    p.learn(1, 1, 0, 0.1)   # y=1 != t=0 -> mise à jour
    assert abs(p.w0 - 0.1) < 1e-9, f"w0 attendu 0.1, obtenu {p.w0}"
    assert abs(p.w1 + 0.1) < 1e-9, f"w1 attendu -0.1, obtenu {p.w1}"
    assert abs(p.w2 + 0.1) < 1e-9, f"w2 attendu -0.1, obtenu {p.w2}"
    p.learn(0, 0, 0, 0.1)   # y=0 == t=0 -> aucun changement
    assert abs(p.w0 - 0.1) < 1e-9
    assert abs(p.w1 + 0.1) < 1e-9
    assert abs(p.w2 + 0.1) < 1e-9
    print("learn check — OK")


check_learn()

### Vérification de `train`

Le perceptron doit apprendre les jeux séparables (entrées 1, 2 et 3) et **échouer** sur le XOR (entrée 4) : après `max_epochs`, il ne sépare toujours pas tous les points.

In [ ]:
def accuracy(p, points):
    ok = sum(1 for pt in points if p.output(pt.x1, pt.x2) == pt.t)
    return ok / len(points)


for name, data in [('entrée 1', entree_1), ('entrée 2', entree_2), ('entrée 3', entree_3)]:
    p = Perceptron(0, 0, 0)
    p.train(data)
    print(f"{name} : accuracy {accuracy(p, data):.0%}  -> {p}")
    assert accuracy(p, data) == 1.0, f"{name} mal apprise"

p_xor = Perceptron(0, 0, 0)
p_xor.train(entree_4, max_epochs=100)
print(f"entrée 4 (XOR) : accuracy {accuracy(p_xor, entree_4):.0%}  -> {p_xor}")
assert accuracy(p_xor, entree_4) < 1.0, "le XOR ne devrait pas être appris"
print("train check — OK")

On peut visualiser l'apprentissage : la droite finit par séparer les points bleus (cible 1) des points orange (cible 0).

In [ ]:
p = Perceptron(0, 0, 0)
p.train(entree_3)
ax = make_axis()
display_plane(p, ax)
display_train(entree_3, ax)
ax.set_title('entrée 3')
plt.show()

## 4. Généralisation (numpy)

On réécrit la classe `Perceptron` pour qu'elle fonctionne en **dimension quelconque** : les poids deviennent un vecteur `w = [w_0, w_1, ..., w_n]` et l'entrée un tableau `x = [x_1, ..., x_n]`. On préfixe l'entrée par $-1$ pour le biais : $s = \langle w, [-1, x_1, \dots, x_n] \rangle$.

> **Exercice 6** — Complétez `output`, `learn` et `train` en version **vectorielle** (numpy).

In [ ]:
@dataclass
class Perceptron:
    w: np.ndarray  # poids [w_0, w_1, ..., w_n] ; w_0 = biais

    def __repr__(self):
        return f"Perceptron({self.w.tolist()})"

    def output(self, x: np.ndarray) -> int:
        # BEGIN
        # raise NotImplementedError
        ## Indice : x_full = np.concatenate(([-1], x)) ; renvoie 1 si w @ x_full >= 0
        x_full = np.concatenate(([-1], x))
        return 1 if float(np.dot(self.w, x_full)) >= 0 else 0
        # END

    def learn(self, x: np.ndarray, t: int, r: float = 0.1):
        # BEGIN
        # raise NotImplementedError
        ## Indice : même règle que la 2D, mais vectorielle
        y = self.output(x)
        if y != t:
            x_full = np.concatenate(([-1], x))
            self.w = self.w + r * (t - y) * x_full
        # END

    def train(self, X: np.ndarray, t: np.ndarray, r: float = 0.1, max_epochs: int = 100):
        # BEGIN
        # raise NotImplementedError
        ## Indice : boucle tant que les poids changent
        for _ in range(max_epochs):
            changed = False
            for x, ti in zip(X, t):
                w_before = self.w.copy()
                self.learn(x, int(ti), r)
                if not np.array_equal(self.w, w_before):
                    changed = True
            if not changed:
                break
        # END

### Vérification de la généralisation

On vérifie en 2D (le ET logique) puis en 3D : le même code doit fonctionner quelle que soit la dimension.

In [ ]:
X_and = np.array([[1, 1], [1, 0], [0, 1], [0, 0]], dtype=float)
t_and = np.array([1, 0, 0, 0])

p = Perceptron(np.zeros(3))
p.train(X_and, t_and)
pred = np.array([p.output(x) for x in X_and])
assert np.array_equal(pred, t_and), "AND mal appris"
print("généralisation (2D, AND) — OK")

X3 = np.array([[1, 1, 1], [1, 1, 0], [0, 0, 0], [0, 0, 1]], dtype=float)
t3 = np.array([1, 1, 0, 0])
p3 = Perceptron(np.zeros(4))
p3.train(X3, t3)
pred3 = np.array([p3.output(x) for x in X3])
assert np.array_equal(pred3, t3), "3D mal appris"
print("généralisation (3D) — OK")